In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
import gc

# 데이터 불러오기

In [3]:
# 기준이 될 한 개 파일로부터 컬럼명만 추출
sample_file = 'train/3.승인매출정보/201807_train_승인매출정보.parquet'
all_columns = pq.ParquetFile(sample_file).schema.names
selected_columns = all_columns[199:]  # 200번째 컬럼부터

# 데이터 병합을 위한 빈 리스트
all_data = []

# 201807 ~ 201812 반복
for month in range(7, 13):
    ym = f"2018{month:02d}"

    # 파일 경로 정의
    train_path = f"train/3.승인매출정보/{ym}_train_승인매출정보.parquet"
    test_path = f"test/3.승인매출정보/{ym}_test_승인매출정보.parquet"

    # 각 파일에서 필요한 컬럼만 읽기
    train_df = pd.read_parquet(train_path, columns=selected_columns)
    test_df = pd.read_parquet(test_path, columns=selected_columns)

    # 해당 월 정보 컬럼 추가
    train_df["year_month"] = ym
    test_df["year_month"] = ym

    # 리스트에 추가
    all_data.extend([train_df, test_df])

# 하나의 데이터프레임으로 병합
all_df = pd.concat(all_data, axis=0, ignore_index=True)

In [4]:
# 결과 확인
all_df

,할부건수_유이자_3M_R12M,할부건수_유이자_6M_R12M,할부건수_유이자_12M_R12M,할부건수_유이자_14M_R12M,할부금액_유이자_3M_R12M,할부금액_유이자_6M_R12M,할부금액_유이자_12M_R12M,할부금액_유이자_14M_R12M,할부건수_무이자_3M_R12M,할부건수_무이자_6M_R12M,...,승인거절건수_BL_B0M,승인거절건수_입력오류_B0M,승인거절건수_기타_B0M,승인거절건수_R3M,승인거절건수_한도초과_R3M,승인거절건수_BL_R3M,승인거절건수_입력오류_R3M,승인거절건수_기타_R3M,이용금액대,year_month
0,0,0,0,0,3602,0,0,0,0,3,...,0,0,0,3,3,0,0,0,01.100만원+,201807
1,0,0,0,0,0,0,0,4058,0,0,...,0,0,0,3,3,0,0,0,03.30만원+,201807
2,0,0,0,0,0,0,0,0,0,3,...,0,0,0,0,0,0,0,0,01.100만원+,201807
3,5,0,0,0,10031,0,0,0,6,0,...,0,0,0,3,3,0,0,0,01.100만원+,201807
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,09.미사용,201807
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,09.미사용,201812
2999996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,05.10만원-,201812
2999997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,09.미사용,201812
2999998,0,0,0,0,0,0,0,0,0,4,...,0,0,0,0,0,0,0,0,01.100만원+,201812


# 명세서 값 채우기

In [ ]:
# 결과 저장 리스트
column_summary = []

# 월 리스트 추출
months = sorted(all_df['year_month'].unique())# 컬럼별 요약
for col in all_df.columns:
    if col == 'year_month':
        continue  # 분석 대상 제외

    data = all_df[col]
    dtype = data.dtype
    example_vals = data.dropna().unique()[:3]

    # 이산형 판단
    is_categorical = (
        dtype == 'object' or 
        dtype.name == 'category' or 
        (data.nunique() < 20 and not np.issubdtype(dtype, np.floating))
    )

    # 연속형 판단
    is_continuous = (
        np.issubdtype(dtype, np.number) and not is_categorical
    )

    # 데이터 유형
    if is_categorical:
        data_type = "범주형"
    elif is_continuous:
        data_type = "수치형"
    else:
        data_type = "기타"

    # 월별 결측치 및 이상치 계산
    missing_by_month = {}
    outlier_by_month = {}

    for m in months:
        sub = all_df[all_df['year_month'] == m][col]

        # 결측치 수
        missing = sub.isnull().sum()
        missing_by_month[m] = missing

        # 이상치 수 (연속형에 한함)
        outlier_count = 0
        if is_continuous and sub.dropna().shape[0] > 0:
            q1 = sub.quantile(0.25)
            q3 = sub.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_count = sub[(sub < lower) | (sub > upper)].count()
        outlier_by_month[m] = outlier_count

    column_summary.append({
        "컬럼명": col,
        "데이터 타입": str(dtype),
        "예시값": example_vals.tolist(),
        "데이터 유형": data_type,
        **{f"{m}_결측": missing_by_month[m] for m in months},
        **{f"{m}_이상치": outlier_by_month[m] for m in months}
    })

# 데이터프레임으로 변환
summary_df = pd.DataFrame(column_summary)

In [ ]:
# 결과확인
summary_df.head()

In [ ]:
#데이터 저장하기
summary_df.to_csv("summary.csv", index=False)